In [1]:
import copy
from pathlib import Path

from vega import BuildConfig

%matplotlib inline

In [ ]:
def make_fit(
    rmin_auto,
    rmin_cross,
    correlations,
    fit_type,
    fit_info,
    params,
    out_path,
    name_start,
    rp_bin_max,
    rt_bin_max,
):
    name_extension = f"{name_start}"
    correlations["lyaxlya"]["r-min"] = rmin_auto
    correlations["lyaxlyb"]["r-min"] = rmin_auto
    correlations["lyaxqso"]["r-min"] = rmin_cross
    correlations["lybxqso"]["r-min"] = rmin_cross

    sample_list = copy.deepcopy(BASE_SAMPLE_PARAMS)
    flat_priors = copy.deepcopy(BASE_FLAT_PRIORS)
    fit_info["priors"] = {}
    for par, prior in BASE_PRIORS.items():
        if par in sample_list:
            fit_info["priors"][par] = prior

    parameters = copy.deepcopy(params)

    options = copy.deepcopy(OPTIONS)
    # options['marginalize-below-rpmax'] = 4 * rp_bin_max
    # options['marginalize-below-rtmax'] = 4 * rt_bin_max

    fit_info["sample_params"] = init_pars(sample_list, flat_priors)

    config_builder = BuildConfig(options, overwrite=True)
    config_builder.build(
        correlations,
        fit_type,
        fit_info,
        out_path,
        parameters=parameters,
        name_extension=name_extension,
    )


def get_correlations(
    path=None,
    qso_cat=None,
    auto_rmin=30,
    cross_rmin=40,
    rmax=180,
    fast_metals=True,
    weights_path=None,
):
    correlations = {"lyaxlya": {}, "lyaxqso": {}, "lyaxlyb": {}, "lybxqso": {}}
    if path is None:
        path = "/global/cfs/cdirs/desicollab/science/lya/y3-fs/validation-tests/v1-4-0-0/baseline"

    deltas_path = Path(path) / "deltas/results/lya/Log/delta_attributes.fits.gz"
    if weights_path is None:
        if not deltas_path.is_file():
            weights_path = Path(
                "/global/cfs/cdirs/desicollab/science/lya/y3-fs/validation-tests/v1-4-0-0/baseline"
            )
        else:
            weights_path = Path(path)
    else:
        weights_path = Path(weights_path)

    if qso_cat is None:
        qso_cat = "/global/cfs/cdirs/desicollab/users/martini/bal-catalogs/loa/QSO_cat_loa_main_dark_healpix_v3-altbal.fits"

    path = Path(path)
    # path2 = '/global/cfs/cdirs/desi/users/abrodze/lya_bias_measurement/variations-1.4/snr3-masked/nhi-20.3/'
    # path3 = '/global/cfs/cdirs/desi/users/abrodze/lya_bias_measurement/variations-1.4/snr3-control/'
    # # '/global/cfs/cdirs/desi/science/lya/y3/loa/correlations/correlation-lyalya-3-0-0/dmat_lya_x_lya.fits'
    # '/global/cfs/cdirs/desi/science/lya/y3/loa/correlations/correlation-qsolya-3-0-0/dmat_qso_x_lya.fits'

    correlations["lyaxlya"]["corr_path"] = str(
        path / "correlations/results/lyalya_lyalya/cf_exp.fits.gz"
    )
    correlations["lyaxlya"]["distortion-file"] = str(
        path / "correlations/results/lyalya_lyalya/dmat.fits.gz"
    )
    correlations["lyaxlya"]["weights-tracer1"] = str(
        weights_path / "deltas/results/lya/Log/delta_attributes.fits.gz"
    )
    correlations["lyaxlya"]["weights-tracer2"] = str(
        weights_path / "deltas/results/lya/Log/delta_attributes.fits.gz"
    )
    correlations["lyaxlya"]["r-min"] = auto_rmin
    correlations["lyaxlya"]["r-max"] = rmax
    correlations["lyaxlya"]["fast_metals"] = f"{fast_metals}"

    correlations["lyaxlyb"]["corr_path"] = str(
        path / "correlations/results/lyalya_lyalyb/cf_exp.fits.gz"
    )
    correlations["lyaxlyb"]["distortion-file"] = str(
        path / "correlations/results/lyalya_lyalyb/dmat.fits.gz"
    )
    correlations["lyaxlyb"]["weights-tracer1"] = str(
        weights_path / "deltas/results/lya/Log/delta_attributes.fits.gz"
    )
    correlations["lyaxlyb"]["weights-tracer2"] = str(
        weights_path / "deltas/results/lyb/Log/delta_attributes.fits.gz"
    )
    correlations["lyaxlyb"]["r-min"] = auto_rmin
    correlations["lyaxlyb"]["r-max"] = rmax
    correlations["lyaxlyb"]["fast_metals"] = f"{fast_metals}"

    correlations["lyaxqso"]["corr_path"] = str(
        path / "correlations/results/qso_lyalya/xcf_exp.fits.gz"
    )
    correlations["lyaxqso"]["distortion-file"] = str(
        path / "correlations/results/qso_lyalya/xdmat.fits.gz"
    )
    correlations["lyaxqso"]["weights-tracer1"] = str(
        weights_path / "deltas/results/lya/Log/delta_attributes.fits.gz"
    )
    correlations["lyaxqso"]["weights-tracer2"] = qso_cat
    correlations["lyaxqso"]["r-min"] = cross_rmin
    correlations["lyaxqso"]["r-max"] = rmax
    correlations["lyaxqso"]["fast_metals"] = f"{fast_metals}"

    correlations["lybxqso"]["corr_path"] = str(
        path / "correlations/results/qso_lyalyb/xcf_exp.fits.gz"
    )
    correlations["lybxqso"]["distortion-file"] = str(
        path / "correlations/results/qso_lyalyb/xdmat.fits.gz"
    )
    correlations["lybxqso"]["weights-tracer1"] = str(
        weights_path / "deltas/results/lyb/Log/delta_attributes.fits.gz"
    )
    correlations["lybxqso"]["weights-tracer2"] = qso_cat
    correlations["lybxqso"]["r-min"] = cross_rmin
    correlations["lybxqso"]["r-max"] = rmax
    correlations["lybxqso"]["fast_metals"] = f"{fast_metals}"

    return correlations


def init_pars(sample_list, flat_priors):
    sample_params = {par: "True" for par in sample_list}
    for par, prior in flat_priors.items():
        if par in sample_params:
            sample_params[par] = prior

    return sample_params

## Customize Vega setup in the cells below

In [3]:
BASE_PRIORS = {
    "dnl_arinyo_q1": "gaussian 1.0 2.0",
    "dnl_arinyo_q2": "gaussian 0.0 1.0",
    "dnl_arinyo_kv": "gaussian 1.0 2.0",
    "dnl_arinyo_av": "gaussian 0.3 0.5",
    "dnl_arinyo_bv": "gaussian 1.6 0.5",
    "dnl_arinyo_kp": "gaussian 14.0 10.0",
    "drp_QSO": "gaussian 0.0 1.0",
    "beta_hcd": "gaussian 0.50 0.09",
    "L0_hcd": "gaussian 5.0 2.0",
    "bias_CIV(eff)": "gaussian -0.019 0.005",
}

BASE_FLAT_PRIORS = {
    # 'bias_hcd': '-0.15 0',
    # 'bias_hcd': '0 10',
    # 'bias_LYA': '-0.2 -0.05',
    # 'beta_LYA': '0.0 15.0',
    "beta_hcd": "0.0 3.0",
    "L0_hcd": "0.0 10.0",
    "dnl_arinyo_q1": "0 2 0.3 0.1",
    "dnl_arinyo_q2": "-1 1 0. 0.1",
    "dnl_arinyo_kv": "0.1 4 1.11454 0.1",
    "dnl_arinyo_av": "0.1 1.0 0.5378 0.1",
    "dnl_arinyo_bv": "1.4 2.0 1.607 0.1",
    "dnl_arinyo_kp": "7.0 22.0 19.47 0.1",
    "bias_gamma": "0.0 1.0 0.1 0.05",
    "lambda_uv": "30 500 300 10",
    "uv_shotnoise_amp": "-1 1 0 0.01",
    "bias_gamma_e": "-1.0 1.0 0.01 0.05",
    "lambda_HeII": "5 100 30 5",
}

OPTIONS = {
    # Parametrization and template
    "scale_params": "ap_at",
    # 'scale_params':'ap_at',
    "template": "Planck18/DESI-2024_z_2.33.fits",
    # This is only needed when fitting correlation functions
    "model-binning": False,
    ##########################
    # Full-shape config; controls scale parameter behaviour
    "full_shape": False,
    "smooth_scaling": False,
    "full_shape_alpha": False,
    ##########################
    # Effects we model at the level of the power spectrum
    # These are ignored when passing an input 2D P(k)
    # NL config
    "small_scale_nl": True,
    "bao_broadening": True,
    "skip-nl-model-in-peak": True,
    # 'fullshape_smoothing': 'gauss',
    "velocity_dispersion": "lorentz",
    "hcd_model": "Rogers2018",
    "UVB-fluctuations": False,
    # 'UVB-SN-cross': False,
    # 'HeII-reionization': False,
    ##########################
    # Small-scale marginalization coonfig
    # 'marginalize-prior-sigma': 10.0,
    "marginalize-all-rmin-cuts": False,
    "fit-marginalized-scales": False,
    "marginalize-match-data-bins": False,
    ##########################
    # Other effects (metals, QSO radiation, sky contamination)
    "use_metal_autos": True,
    "new_metals": True,
    "metals": "all",
    "metal-matrix": {"alpha_CIV(eff)": "0"},
    "rp_only_metal_mats": True,
    "radiation_effects": True,
    "desi-instrumental-systematics": True,
}


BASE_SAMPLE_PARAMS = [
    "bias_LYA",
    "beta_LYA",
    "bias_QSO",
    "sigma_velo_disp_lorentz_QSO",
    "drp_QSO",
    "qso_rad_strength",
    "bias_SiII(1190)",
    "bias_SiII(1193)",
    "bias_SiIII(1207)",
    "bias_SiII(1260)",
    "bias_CIV(eff)",
    "desi_inst_sys_amp",
]

## Modify the name and out_path in this cell

In [4]:
name = "example_v0"
out_path = "/global/cfs/projectdirs/desi/users/acuceu/vega_dev/p3d/configs/"

fit_info = {
    "run_sampler": True,
    "zeff": 2.33,
    "zeff_rmin": -300.0,
    "zeff_rmax": 300.0,
    "bias_beta_config": {"LYA": "bias_beta", "QSO": "bias_bias_eta"},
    "use_template_growth_rate": "False",
    "low_mem_mode": "False",
    "Polychord": {"num_live": "512", "num_repeats": "24"},
    "priors": {},
    # 'global_cov_file': '/global/cfs/cdirs/desicollab/science/lya/y3-fs/validation-tests/v1-4-0-0/baseline/fits/results/full-covariance-smoothed.fits',
}

correlations = get_correlations(auto_rmin=30, cross_rmin=40, rmax=200, fast_metals=True)

parameters = {
    "bias_LYA": -0.15,
    "beta_LYA": 1.5,
    "drp_QSO": 0.0,
    "bias_QSO": 3.4,
    "sigma_velo_disp_lorentz_QSO": 4.2,
    "bias_SiIII(1207)": -0.00979,
}

## Choose the combination of correlations to model and create configs

In [ ]:
######## Fit ########

# fit_type = 'lyaxlya_lyaxlyb_lyaxqso_lybxqso'
fit_type = "lyaxlya_lyaxqso"
# fit_type = 'lyaxlya'
# fit_type = 'lyaxqso'
if OPTIONS["rp_only_metal_mats"]:
    print("WARNING: RP only metal mats turned on!!!")

rmin_auto = 30
rmin_cross = 40
make_fit(rmin_auto, rmin_cross, correlations, fit_type, fit_info, parameters, out_path, name, 0, 0)